# Quant Strategy Sandbox
Template for testing individual trading signals and candidate models.
Deadline for candidate models: **Thursday 12 Nov (23:59)**.

In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().resolve().parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import polars as pl
from src.utils import load_prices, evaluate_model, plot_model, check_causality, split_walk_forward, DATE

prices = load_prices()
instruments = [c for c in prices.columns if c != DATE]
print(f'Loaded {len(prices)} bars. Instruments ({len(instruments)}): {instruments}')

## 1. Define Candidate Strategy
Input: `ret` (price difference returns, $P_T - P_{T-1}$).
Output: `positions` (DataFrame with `AsOfDate` and one column per traded asset).

In [ ]:
def candidate_strategy(ret: pl.DataFrame) -> pl.DataFrame:
    cols = [c for c in ret.columns if c != DATE]
    
    # 1. Realized price volatility (60 days)
    vol = ret.select(DATE, pl.col(cols).pow(2).rolling_sum(60, min_samples=30).sqrt())
    
    # 2. Directional signal (multi-horizon tanh momentum)
    s_fast = ret.select(DATE, pl.col(cols).rolling_sum(30))
    s_slow = ret.select(DATE, pl.col(cols).rolling_sum(80))
    signal = ((0.4 * s_fast + 0.6 * s_slow) / vol).tanh()
    
    # 3. Position sizing (inverse vol scaling in cash units)
    positions = signal.select(DATE, *[(pl.col(c) / vol[c]).alias(c) for c in cols])
    
    # 4. Portfolio model risk normalization
    gross = positions.select(DATE, *[(pl.col(c).shift(2) * ret[c]).alias(c) for c in cols])
    risk = gross.filter(~pl.all_horizontal(pl.col(cols).is_null())).select(
        DATE, pl.sum_horizontal(pl.col(cols)).rolling_std(100, min_samples=20).alias('model_risk')
    )
    aligned = positions.join(risk, on=DATE, how='left', maintain_order='left')
    return aligned.select(DATE, *[(pl.col(c) / pl.col('model_risk')).fill_nan(None).alias(c) for c in cols])

In [ ]:
# 2. Causality Check (Zero Lookahead Proof)
passed, max_diff = check_causality(candidate_strategy, prices)
print(f'Causality test: {"PASSED (Zero Lookahead)" if passed else "FAILED"} (max diff: {max_diff:.2e})')
assert passed, 'Disqualified for forward-looking leakage!'

In [ ]:
# 3. Official Evaluation Scorecard
ret = prices.select(DATE, pl.col(instruments).diff())
positions = candidate_strategy(ret)
res = evaluate_model(positions, prices)

print('=' * 42)
print(f"Adjusted Sharpe  : {res['adjusted_sharpe']:.4f}")
print(f"Nominal Sharpe   : {res['sharpe']:.4f}")
print(f"Sharpe (+1d lag) : {res['sharpe_1d_lag']:.4f}")
print(f"Average Holding  : {res['av_holding']:.1f} days")
print(f"Position Bias    : {res['position_bias']:.4f}")
print(f"Excess Kurtosis  : {res['kurtosis'] - 3.0:.4f}")
print('=' * 42)

In [ ]:
# 4. Official Diagnostic Charts
plot_model(positions, prices)